<a href="https://colab.research.google.com/github/marygizem/-Machine-Learning-and-Predictive-Analytics/blob/main/Updated_Milestone_2_Project_Forecasting_Model_with_DataViz_Final_Google_Collab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Milestone 2: Project Forecasting Model with DataViz

## Project Overview

FinMark Corporation currently relies on manual methods for analyzing sales data and preparing forecasts. While these traditional processes may work for small datasets, they become inefficient as the company grows and handles a larger volume of transactions. Manual forecasting can lead to slow reporting, limited visibility into sales trends, and delayed responses to changes in customer demand and market conditions.

To address this challenge, this project applies data analytics and machine learning techniques to automate sales forecasting using historical transaction data. The goal is to transform raw business data into meaningful insights that can support better planning and faster decision-making.

In this Milestone 2 project, the focus is on forecasting FinMark Corporation’s sales for the next six months. The analysis involves preparing and aggregating historical sales data, engineering forecasting features, training and comparing multiple models, and visualizing both past and projected sales patterns.

By automating the forecasting process, FinMark Corporation can improve efficiency, identify trends more quickly, and make more informed business decisions based on data-driven projections.




# Importing Libraries and Loading the Dataset

This section imports the for data analysis, modeling, and visualization. It also loads the dataset that contains the historical sales records used in this forecasting task.

The main libraries used are:
- **pandas** for data manipulation
- **numpy** for numerical operations
- **matplotlib** for visualization
- **scikit-learn** for machine learning models and evaluation metrics



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.arima.model import ARIMA

## Colab File Setup

If `merged_data.csv` is not already in your Colab session, run the next cell to upload it manually. This makes the notebook easier to open and run in Google Colab.


In [ ]:
from google.colab import files
import os

if not os.path.exists("merged_data.csv"):
    print("Upload merged_data.csv to continue.")
    uploaded = files.upload()
else:
    print("merged_data.csv is already available.")


Upload merged_data.csv to continue.


##Understanding the Dataset

This section checks:
- the first few rows of the dataset
- the column names
- the data types of each column
- whether there are missing values
- whether the date column is in the correct format



In [ ]:
# ============================================================
# 2) LOAD AND INSPECT THE DATA
# ============================================================
"""
Documentation:
- The dataset is loaded from merged_data.csv.
- Transaction_Date is converted to datetime.
- Data is aggregated by month because the forecasting task is monthly sales prediction.
"""

merged_data = pd.read_csv("merged_data.csv")
merged_data["Transaction_Date"] = pd.to_datetime(merged_data["Transaction_Date"])

print("DATA LOADED SUCCESSFULLY")
print(merged_data.head())

##Monthly Sales Aggregation

The original transaction-level dataset is transformed into monthly sales totals so that the forecasting model can learn from time-based sales patterns.

This step groups the data by month and calculates the total sales for each month. Monthly aggregation is appropriate because the objective of the project is to forecast sales on a monthly basis rather than predict individual transactions.

The result of this section is a simplified time series dataset with:
- one row per month
- a corresponding total sales value for that month



In [ ]:
# ============================================================
# 3) MONTHLY SALES AGGREGATION
# ============================================================
"""
Documentation:
- Transaction-level records are grouped into monthly totals.
- Additional monthly business indicators are also computed:
  * total_quantity
  * avg_price
  * transaction_count
"""

monthly_sales = (
    merged_data.groupby(merged_data["Transaction_Date"].dt.to_period("M"))
    .agg(
        total_sales=("Total_Cost", "sum"),
        total_quantity=("Quantity", "sum"),
        avg_price=("Product_Price_x", "mean"),
        transaction_count=("Transaction_ID", "count"),
    )
    .reset_index()
)

monthly_sales["YearMonth"] = monthly_sales["Transaction_Date"].dt.to_timestamp()
monthly_sales = monthly_sales.drop(columns=["Transaction_Date"]).sort_values("YearMonth").reset_index(drop=True)

print("\nMONTHLY SALES AGGREGATION")
print(monthly_sales.head())
print("Shape:", monthly_sales.shape)

## Handling Extreme Outliers in Monthly Sales

This section identifies and handles extreme monthly sales values that may distort the forecasting results.

Outliers are capped using the Interquartile Range (IQR) method so that unusually large sales spikes do not cause unrealistic future forecasts. The original sales values are preserved, while a cleaned version is used for model training.

This step improves forecast stability and directly addresses the feedback that some forecasted values appeared unusual.
"""



In [ ]:
# ============================================================
# 4) HANDLE EXTREME OUTLIERS IN MONTHLY SALES
# ============================================================
"""
Documentation:
- One reason forecasted values may look unusual is the presence of extreme monthly outliers.
- To stabilize the model, monthly sales are capped using the IQR method.
- The original total_sales column is preserved, while total_sales_used is the cleaned target.
"""

q1 = monthly_sales["total_sales"].quantile(0.25)
q3 = monthly_sales["total_sales"].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

monthly_sales["total_sales_used"] = monthly_sales["total_sales"].clip(lower=lower_bound, upper=upper_bound)

outliers = monthly_sales[monthly_sales["total_sales"] != monthly_sales["total_sales_used"]][["YearMonth", "total_sales", "total_sales_used"]]

print("\nOUTLIER CHECK")
if len(outliers) > 0:
    print("The following monthly values were capped to reduce distortion:")
    print(outliers)
else:
    print("No extreme outliers detected.")

## Historical Sales Visualization

This section presents a line chart of the historical monthly sales data.

The purpose of this visualization is to:
- show the overall sales trend over time
- identify seasonal or recurring patterns
- detect unusual spikes or drops in monthly sales
- provide a clear view of the dataset before forecasting begins

Visualizing the historical sales data is important because it helps verify whether the time series behaves consistently and whether there are extreme values that may affect model performance. This also supports the later decision to handle outliers before training the forecasting models.

By examining the chart, the reader can better understand the sales pattern and assess whether the forecasted results are reasonable when compared with past performance.
"""

In [ ]:
# ============================================================
# 5) HISTORICAL SALES VISUALIZATION
# ============================================================
"""
Documentation:
- The first plot shows the original monthly sales trend.
- The second plot shows the cleaned monthly sales used for modeling.
- This makes the data preparation step transparent.
"""

plt.figure(figsize=(11, 5))
plt.plot(monthly_sales["YearMonth"], monthly_sales["total_sales"], marker="o")
plt.title("Original Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Total Sales")
plt.grid(True)
plt.show()

plt.figure(figsize=(11, 5))
plt.plot(monthly_sales["YearMonth"], monthly_sales["total_sales_used"], marker="o")
plt.title("Cleaned Monthly Sales Trend Used for Forecasting")
plt.xlabel("Month")
plt.ylabel("Total Sales")
plt.grid(True)
plt.show()


## Feature Engineering

New features were created to improve forecasting accuracy. These include:

- Time-based features (month and quarter)
- Lag features (previous sales values)
- Rolling average (moving average of past sales)

These features allow the model to learn patterns from past behavior and improve prediction performance.

In [ ]:
# ============================================================
# 6) FEATURE ENGINEERING
# ============================================================
"""
Documentation:
- Time-based and lag-based features are created to help machine learning models learn patterns.
- Features:
  * month
  * quarter
  * year
  * lag_1, lag_2, lag_3
  * rolling_3, rolling_6
  * total_quantity
  * avg_price
  * transaction_count
"""

monthly_sales["month"] = monthly_sales["YearMonth"].dt.month
monthly_sales["quarter"] = monthly_sales["YearMonth"].dt.quarter
monthly_sales["year"] = monthly_sales["YearMonth"].dt.year

for lag in [1, 2, 3]:
    monthly_sales[f"lag_{lag}"] = monthly_sales["total_sales_used"].shift(lag)

monthly_sales["rolling_3"] = monthly_sales["total_sales_used"].shift(1).rolling(window=3).mean()
monthly_sales["rolling_6"] = monthly_sales["total_sales_used"].shift(1).rolling(window=6).mean()

model_data = monthly_sales.dropna().reset_index(drop=True)

print("\nFEATURE ENGINEERING OUTPUT")
print(model_data.head())
print("Shape:", model_data.shape)



##Train-Test Preparation

This section prepares the dataset for forecasting by separating it into training and testing sets.

The last six months are reserved for testing, while the earlier observations are used for training. This approach follows the correct time order of the data and simulates a real forecasting scenario where past data is used to predict future values.


In [ ]:
# 7) TRAIN-TEST SPLIT
# ============================================================
"""
Documentation:
- The last 6 months are reserved as the test set.
- Earlier months are used for training.
- This simulates a real forecasting setup.
"""

feature_cols = [
    "month", "quarter", "year",
    "lag_1", "lag_2", "lag_3",
    "rolling_3", "rolling_6",
    "total_quantity", "avg_price", "transaction_count"
]

train = model_data.iloc[:-6].copy()
test = model_data.iloc[-6:].copy()

X_train = train[feature_cols]
y_train = train["total_sales_used"]
X_test = test[feature_cols]
y_test = test["total_sales_used"]

print("\nTRAIN SHAPE:", X_train.shape)
print("TEST SHAPE:", X_test.shape)

## Model Training and Comparison

This section trains multiple forecasting models using the prepared training dataset and compares their performance using the test dataset.

Evaluate different algorithms and determine which model produces the most accurate and most reasonable forecasts.

The models used in this analysis include:
- Linear Regression
- Ridge Regression
- Random Forest Regressor
- Gradient Boosting Regressor
- ARIMA

Each model is trained using the historical sales features, then evaluated using error metrics such as Mean Absolute Error (MAE) and Root Mean Squared Error (RMSE). Lower values for these metrics indicate better forecasting performance.


In [ ]:
# ============================================================
# 8) MODEL TRAINING AND COMPARISON
# ============================================================
"""
Documentation:
- Multiple algorithms are tested:
  * Linear Regression
  * Ridge Regression
  * Random Forest Regressor
  * Gradient Boosting Regressor
  * ARIMA
- Performance is compared using MAE and RMSE.
- Lower values indicate better forecasting performance.
"""

models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        max_depth=8,
        random_state=42
    ),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
}

comparison_rows = []
prediction_store = {}

for model_name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))

    comparison_rows.append([model_name, mae, rmse])
    prediction_store[model_name] = preds

In [ ]:
# ARIMA baseline
arima_model = ARIMA(train["total_sales_used"], order=(1, 1, 1))
arima_fit = arima_model.fit()
arima_preds = arima_fit.forecast(steps=len(test))

arima_mae = mean_absolute_error(y_test, arima_preds)
arima_rmse = np.sqrt(mean_squared_error(y_test, arima_preds))
comparison_rows.append(["ARIMA", arima_mae, arima_rmse])
prediction_store["ARIMA"] = np.array(arima_preds)

comparison_df = pd.DataFrame(comparison_rows, columns=["Model", "MAE", "RMSE"]).sort_values("RMSE").reset_index(drop=True)

print("\nMODEL COMPARISON")
print(comparison_df)

best_model_name = comparison_df.iloc[0]["Model"]
print("\nBEST MODEL:", best_model_name)


## Model Evaluation

After training, the models are evaluated using the test set. Their predictions are compared with the actual sales values from the last six months.

The following evaluation metrics are used:

- **MAE (Mean Absolute Error)**  
  This measures the average absolute difference between actual and predicted sales values. A lower MAE means the model makes smaller errors on average.

- **RMSE (Root Mean Squared Error)**  
  This measures the square root of the average squared prediction errors. RMSE gives more weight to larger errors, making it useful for detecting models that produce highly inaccurate forecasts.

These metrics make it possible to compare models objectively and select the best-performing one for the final six-month forecast.

In [ ]:
# ============================================================
# 9) FORECAST TABLE FOR TEST PERIOD
# ============================================================
"""
Documentation:
- This table shows actual sales and the forecasted values for the holdout period.
- It directly addresses the requirement to display forecasted values clearly.
"""

results = pd.DataFrame({
    "Month": test["YearMonth"].dt.strftime("%Y-%m"),
    "Actual Sales": y_test.values,
    "Linear Regression Forecast": prediction_store["Linear Regression"],
    "Ridge Regression Forecast": prediction_store["Ridge Regression"],
    "Random Forest Forecast": prediction_store["Random Forest"],
    "Gradient Boosting Forecast": prediction_store["Gradient Boosting"],
    "ARIMA Forecast": prediction_store["ARIMA"],
})

print("\nACTUAL VS FORECASTED VALUES")
print(results)

## Model Comparison

This section summarizes the performance of all trained models in a comparison table.

The purpose of the comparison table is to:
- clearly show the MAE and RMSE of each model
- identify the best-performing model
- justify why one model is selected over another for future forecasting

The model with the lower RMSE is selected as the best model because RMSE penalizes larger prediction errors more strongly. This makes it especially useful in forecasting tasks where extreme mistakes can distort business decisions.

In [ ]:
# ============================================================
# 10) MODEL COMPARISON VISUALIZATION
# ============================================================
"""
Documentation:
- This chart compares the RMSE of all evaluated models.
- The best model is the one with the lowest RMSE.
"""

plt.figure(figsize=(10, 5))
plt.bar(comparison_df["Model"], comparison_df["RMSE"])
plt.title("Model Comparison Using RMSE")
plt.xlabel("Model")
plt.ylabel("RMSE")
plt.xticks(rotation=20)
plt.grid(True)
plt.show()

##  Actual vs Forecasted Values

This section displays a table containing:
- the month
- the actual sales value
- the predicted sales value from Linear Regression
- the predicted sales value from Random Forest

This section displays the actual sales values and the predicted sales values from all evaluated models during the test period.



In [ ]:
# ============================================================
# 11) ACTUAL VS PREDICTED VISUALIZATION FOR BEST MODEL
# ============================================================
"""
Documentation:
- This graph compares actual sales with the predictions of the best-performing model
  on the last 6 months of known data.
"""

best_test_preds = prediction_store[best_model_name]

plt.figure(figsize=(11, 5))
plt.plot(test["YearMonth"], y_test.values, marker="o", label="Actual Sales")
plt.plot(test["YearMonth"], best_test_preds, marker="o", label=f"{best_model_name} Forecast")
plt.title(f"Actual vs Forecasted Sales ({best_model_name})")
plt.xlabel("Month")
plt.ylabel("Sales")
plt.legend()
plt.grid(True)
plt.show()

## Fit the Best ML Model on All Available Data

After comparing the performance of the forecasting models, the best-performing machine learning model is selected and retrained using all available historical data.

This step is important because once the best model has been identified, it should learn from the complete dataset instead of only the training portion. Using all available observations allows the model to capture more information from the sales history before generating future forecasts.



In [ ]:
# ============================================================
# 12) FIT THE BEST ML MODEL ON ALL AVAILABLE MODEL DATA
# ============================================================
"""
Documentation:
- For the final 6-month future forecast, the best machine learning model is retrained
  on all available historical observations.
- ARIMA is excluded from the recursive feature-based forecasting section because the
  future feature generation below is designed for supervised ML models.
"""

ml_models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        max_depth=8,
        random_state=42
    ),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
}

if best_model_name == "ARIMA":
    best_model_name = "Ridge Regression"

best_model = ml_models[best_model_name]
best_model.fit(model_data[feature_cols], model_data["total_sales_used"])

## Six-Month Forecast Generation

It is used to generate sales forecasts for the next six months beyond the available historical dataset.

This forecast is created using a **recursive forecasting approach**. In this method:
- the first future month is predicted using the latest available historical values
- that predicted value is then added back into the dataset
- the next month is predicted using both actual past values and previous forecasted values
- the process repeats until six future months are generated

This approach is commonly used in time series forecasting when future actual values are not yet available.



In [ ]:
# ============================================================
# 13) SIX-MONTH FUTURE FORECAST
# ============================================================
"""
Documentation:
- Forecasting is done recursively for the next 6 months.
- Each new prediction becomes part of the input for the next month.
- This section prints the future forecast values and shows the required forecast plots.
"""

future_rows = []
future_working = monthly_sales.copy()

for _ in range(6):
    next_month = future_working["YearMonth"].iloc[-1] + pd.DateOffset(months=1)

    next_row = {
        "YearMonth": next_month,
        "month": next_month.month,
        "quarter": next_month.quarter,
        "year": next_month.year,
        "lag_1": future_working["total_sales_used"].iloc[-1],
        "lag_2": future_working["total_sales_used"].iloc[-2],
        "lag_3": future_working["total_sales_used"].iloc[-3],
        "rolling_3": future_working["total_sales_used"].iloc[-3:].mean(),
        "rolling_6": future_working["total_sales_used"].iloc[-6:].mean(),
        "total_quantity": future_working["total_quantity"].iloc[-6:].mean(),
        "avg_price": future_working["avg_price"].iloc[-6:].mean(),
        "transaction_count": future_working["transaction_count"].iloc[-6:].mean(),
    }

    next_features = pd.DataFrame([next_row])[feature_cols]
    next_prediction = best_model.predict(next_features)[0]

    next_row["total_sales_used"] = next_prediction
    future_rows.append({
        "Month": next_month.strftime("%Y-%m"),
        "Forecasted Sales": next_prediction
    })

    future_working = pd.concat([future_working, pd.DataFrame([next_row])], ignore_index=True)

future_forecast = pd.DataFrame(future_rows)

print("\nSIX-MONTH FORECAST")
print(future_forecast)

## Visualization of the Six-Month Forecast

This section presents the projected sales values using charts to make the forecast easier to interpret.

The visualizations include:
- a line chart of the six-month forecast
- a combined historical and forecasted sales chart
- a bar chart of projected sales

These charts help the reader understand:
- the expected direction of future sales
- whether projected sales are increasing, decreasing, or stable
- how the forecast compares to historical sales patterns



In [ ]:
# ============================================================
# 14) SIX-MONTH FORECAST VISUALIZATIONS
# ============================================================

plt.figure(figsize=(10, 5))
plt.plot(future_forecast["Month"], future_forecast["Forecasted Sales"], marker="o")
plt.title(f"Six-Month Sales Forecast Using {best_model_name}")
plt.xlabel("Month")
plt.ylabel("Forecasted Sales")
plt.grid(True)
plt.show()

historical_plot = monthly_sales[["YearMonth", "total_sales_used"]].copy()
historical_plot["Type"] = "Historical"

forecast_plot = future_forecast.copy()
forecast_plot["YearMonth"] = pd.to_datetime(forecast_plot["Month"])
forecast_plot["total_sales_used"] = forecast_plot["Forecasted Sales"]
forecast_plot["Type"] = "Forecast"

plt.figure(figsize=(12, 6))
plt.plot(historical_plot["YearMonth"], historical_plot["total_sales_used"], marker="o", label="Historical Sales")
plt.plot(forecast_plot["YearMonth"], forecast_plot["total_sales_used"], marker="o", label="Forecasted Sales")
plt.title("Historical and Forecasted Sales (Next 6 Months)")
plt.xlabel("Month")
plt.ylabel("Sales")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 5))
plt.bar(future_forecast["Month"], future_forecast["Forecasted Sales"])
plt.title("Projected Sales for the Next 6 Months")
plt.xlabel("Month")
plt.ylabel("Forecasted Sales")
plt.xticks(rotation=45)
plt.grid(True)
plt.show()


## Interpretation


The results of this forecasting analysis show that historical transaction data can be transformed into meaningful sales projections through data preparation, feature engineering, and machine learning. By aggregating transaction records into monthly sales totals, the dataset became suitable for time-based forecasting and trend analysis.

The comparison of multiple models helped identify which algorithm produced the most reliable predictions for the available sales data. Using evaluation metrics such as MAE and RMSE made the model selection process more objective, while the forecast tables and visualizations made the results easier to interpret and validate.

The historical sales charts also revealed that the dataset contains fluctuations and some unusually high monthly sales values. Because of this, handling extreme values and comparing several models were important steps in producing more stable and reasonable forecasts.

Overall, the final forecasting workflow provides FinMark Corporation with a more efficient and data-driven approach to sales forecasting. Instead of relying on manual estimation, the company can use projected sales patterns to support planning, budgeting, and decision-making over the next six months.



## Conclusion

This notebook successfully developed a sales forecasting workflow for FinMark Corporation using historical transaction data.

The analysis included monthly sales aggregation, outlier handling, feature engineering, model training and comparison, forecast evaluation, and six-month future sales forecasting with visualizations.

This revised version addresses the feedback by providing detailed documentation, clearly showing forecasted values, comparing multiple algorithms, and including complete data visualizations.